# RAG (Retrieval-Augmented Generation)

**RAG** is a technique used in AI to make answers **more accurate, reliable, and up-to-date** by letting the model **retrieve real information first**, then generate an answer using that information.

---

## 🔹 Why RAG exists

Normally, an AI:
- Answers only from what it learned during training
- May have outdated or incomplete knowledge
- Can confidently guess when unsure (hallucination)

**RAG fixes this by allowing the AI to look things up before answering.**

Think of it like:
> *“Check the notes first, then explain.”*

---

## 🔹 What the name means

### 1. Retrieval
The system searches for relevant information from:
- Documents (PDFs, text files)
- Databases
- Knowledge bases
- Internal company data

### 2. Augmented
The retrieved information is **added to the AI’s prompt** as extra context.

### 3. Generation
The AI generates a natural-language answer using:
- Its language abilities
- The retrieved information

---

## 🔹 Simple analogy

**Without RAG:**  
A student answers from memory only.

**With RAG:**  
A student:
1. Goes to the library
2. Reads the right pages
3. Explains the answer clearly

---



## 🔹 Step-by-step: How RAG works

Example question:
> *“What is our company’s vacation policy?”*

1. **User asks a question**
2. **Retrieval:** System finds relevant documents (HR policy)
3. **Augmentation:** Relevant text is added to the prompt
4. **Generation:** AI explains the answer using that information

---

## 🔹 Benefits of RAG

- ✅ More accurate answers
- ✅ Uses up-to-date information
- ✅ Fewer hallucinations
- ✅ Works with private or custom data
- ✅ No need to retrain the model

---

## 🔹 What RAG is NOT

- ❌ It does NOT train the model
- ❌ It does NOT permanently store data
- ❌ It does NOT change model weights

RAG simply provides **temporary context** at query time.

---

## 🔹 Common use cases

- Chatbots over documents
- Customer support systems
- Company knowledge assistants
- Legal or medical Q&A systems
- Research assistants

If you’ve seen **“Chat with your documents”**, that is RAG.

---

## 🔹 One-line summary

**RAG allows an AI to retrieve relevant information first and then generate a better, more accurate answer.**


### Expert Knowledge Worker

- A question answering Assistant that is an expert knowledge worker
- To be used by employees of Insurellm, an Insurance Tech company
- The AI assistant needs to be accurate and the solution should be low cost.
- This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simplistic, brute-force type of RAG

RAG is perhaps the most immediately applicable technique for Business. In fact, there are commercial products that do precisely what we build this Chapter: nuanced querying across large databases of information, such as company contracts or product specs. RAG gives you a quick-to-market, low cost mechanism for adapting an LLM to your business area.

In [3]:
# Importing Libraries

import os
import json
import time
import re
import glob
import requests
import ollama
from pathlib import Path
import gradio as gr

#### Let's read in all employee data into a dictionary

In [52]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME="gemma3"

In [53]:
# Loading some fake data with MD format

knowledge = {}

filenames = glob.glob("/home/alexender/Desktop/Projects/My_projects/Data/knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [54]:
knowledge['spencer']

'# HR Record\n\n# Oliver Spencer\n\n## Summary\n- **Date of Birth**: May 14, 1990\n- **Job Title**: Backend Software Engineer\n- **Location**: Austin, Texas\n- **Current Salary**: $125,000  \n\n## Insurellm Career Progression\n- **March 2018**: Joined Insurellm as a Backend Developer I, focusing on API development for customer management systems.\n- **July 2019**: Promoted to Backend Developer II after successfully leading a team project to revamp the claims processing system, reducing response time by 30%.\n- **June 2021**: Transitioned to Backend Software Engineer with a broader role in architecture and system design, collaborating closely with the DevOps team.\n- **September 2022**: Assigned as the lead engineer for the new "Innovate" initiative, aimed at integrating AI-driven solutions into existing products.\n- **January 2023**: Awarded a mentorship role to guide new hires in backend technology and best practices within Insurellm.\n\n## Annual Performance History\n- **2018**: **3/

In [55]:
filenames = glob.glob("/home/alexender/Desktop/Projects/My_projects/Data/knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [56]:
knowledge.keys()

dict_keys(['spencer', 'kim', 'chen', 'thompson', 'bishop', 'anderson', 'carter', 'trenton', 'greene', 'zhang', 'thomson', 'brooks', 'wilson', 'sharma', 'harper', 'walker', 'park', 'liu', 'rodriguez', 'lancaster', 'blake', 'rivera', 'patel', 'foster', 'williams', 'johnson', 'martinez', 'adams', "o'brien", 'tran', 'carllm', 'markellm', 'rellm', 'homellm', 'healthllm', 'claimllm', 'bizllm', 'lifellm'])

In [57]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [58]:
def get_relevent_context_simple(messages):
    text = "".join(ch for ch in messages if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

In [59]:
from IPython.display import Markdown

Markdown(get_relevent_context_simple("Who is Lancaster and what is carllm?")[0])

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.

## Annual Performance History
- **2015**: **Exceeds Expectations**  
  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  

- **2016**: **Meets Expectations**  
  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  

- **2017**: **Developing**  
  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  

- **2018**: **Exceeds Expectations**  
  Under Avery’s pivoted vision, Insurellm launched two new successful products that significantly increased market share.  

- **2019**: **Meets Expectations**  
  Steady growth, however, some team tensions led to a minor drop in employee morale. Avery recognized the need to enhance company culture.  

- **2020**: **Below Expectations**  
  The COVID-19 pandemic posed unforeseen operational difficulties. Avery faced criticism for delayed strategy shifts, although efforts were eventually made to stabilize the company.  

- **2021**: **Exceptional**  
  Avery's decisive transition to remote work and rapid adoption of digital tools led to record-high customer satisfaction levels and increased sales.  

- **2022**: **Satisfactory**  
  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  

- **2023**: **Exceeds Expectations**  
  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.

## Compensation History
- **2015**: $150,000 base salary + Significant equity stake  
- **2016**: $160,000 base salary + Equity increase  
- **2017**: $150,000 base salary + Decrease in bonus due to performance  
- **2018**: $180,000 base salary + performance bonus of $30,000  
- **2019**: $185,000 base salary + market adjustment + $5,000 bonus  
- **2020**: $170,000 base salary (temporary reduction due to COVID-19)  
- **2021**: $200,000 base salary + performance bonus of $50,000  
- **2022**: $210,000 base salary + retention bonus  
- **2023**: $225,000 base salary + $75,000 performance bonus  

## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  
- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.
- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  

Avery Lancaster has demonstrated resilience and adaptability throughout her career at Insurellm, positioning the company as a key player in the insurance technology landscape.

In [60]:
Markdown(get_relevant_context("Who is Lancaster and what is carllm?")[1])

# Product Summary

# Carllm

## Summary

Carllm is an innovative auto insurance product developed by Insurellm, designed to streamline the way insurance companies offer coverage to their customers. Powered by cutting-edge artificial intelligence, Carllm utilizes advanced algorithms to deliver personalized auto insurance solutions, ensuring optimal coverage while minimizing costs. With a robust infrastructure that supports both B2B and B2C customers, Carllm redefines the auto insurance landscape and empowers insurance providers to enhance customer satisfaction and retention.

## Features

- **AI-Powered Risk Assessment**: Carllm leverages artificial intelligence to analyze driver behavior, vehicle conditions, and historical claims data. This enables insurers to make informed decisions and set competitive premiums that reflect true risk profiles.

- **Instant Quoting**: With Carllm, insurance companies can offer near-instant quotes to customers, enhancing the customer experience. The AI engine processes data in real-time, drastically reducing the time it takes to generate quotes.

- **Customizable Coverage Plans**: Carllm allows insurers to create flexible and tailored insurance packages based on individual customer needs. This customization improves customer engagement and retention.

- **Fraud Detection**: The product incorporates advanced analytics to identify potentially fraudulent claims, significantly reducing the risk of losses for insurance providers.

- **Customer Insights Dashboard**: Carllm provides insurers with a powerful dashboard that offers deep insights into customer behavior, claims patterns, and market trends, enabling informed decision-making and strategic planning.

- **Mobile Integration**: Carllm is designed to work seamlessly with mobile applications, providing both insurers and end-users access to policy management and claims reporting on the go.

- **Automated Customer Support**: Leveraging AI chatbots, Carllm offers 24/7 customer support, helping to resolve inquiries quickly and efficiently, thus improving customer satisfaction.

## Pricing

Carllm is offered under a subscription-based pricing model tailored to meet the needs of insurance companies of all sizes. Our pricing tiers are designed to provide maximum flexibility and value:

- **Basic Tier**: $1,000/month
  - Ideal for small insurance firms.
  - Access to core features and standard reporting.

- **Professional Tier**: $2,500/month
  - For medium-sized companies.
  - All Basic Tier features plus advanced analytics and fraud detection.

- **Enterprise Tier**: $5,000/month
  - Customized solutions for large insurance firms.
  - Comprehensive support, full feature access, and integration with existing systems.

Contact our sales team for a personalized quote and discover how Carllm can transform your auto insurance offerings!

## 2025-2026 Roadmap

In our commitment to continuous improvement and innovation, Insurellm has outlined the following roadmap for Carllm:

### Q1 2025: Launch Feature Enhancements
- **Expanded data integrations** for better risk assessment.
- **Enhanced fraud detection algorithms** to reduce losses.

### Q2 2025: Customer Experience Improvements
- Launch of a new **mobile app** for end-users.
- Introduction of **telematics-based pricing** to provide even more tailored coverage options.

### Q3 2025: Global Expansion
- Begin pilot programs for international insurance markets.
- Collaborate with local insurers to offer compliant, localized versions of Carllm.

### Q4 2025: AI and Machine Learning Upgrades
- Implement next-gen machine learning models for predictive analysis.
- Roll out customer insights dashboard updates based on user feedback.

### 2026: Scaling and Partnerships
- Increase partnerships with automakers for integrated insurance solutions.
- Enhance the **AI customer support system** to include multi-language support.

Carllm is not just an auto insurance product; it is a transformative tool for the insurance industry. Join us on this exciting journey as we redefine the future of auto insurance with technology and customer-centric solutions.

In [61]:
def additional_context(messages):
    relevant_context = get_relevent_context_simple(messages)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [62]:
print(additional_context("Who is Alex Lancaster?"))

The following additional context might be relevant in answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joini

In [63]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            stream=False,
            options={
                "temperature": 0.8
            }
        )
        # print(response, "\n")

        msg = response.get("message", {})
        if msg.get("content"):
            return {"role": "assistant", "content": msg["content"]}
        
        return {"role": "assistant", "content": "I'm not sure how to respond."}
                    
    except Exception as e:
        print("Exception: ", repr(e))

##### Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [64]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
